In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load dataset voice.csv
GITHUB_URL = 'https://raw.githubusercontent.com/oninonins/Machine-learning-/main/JS09/voice.csv'
CANDIDATES = [
    'voice.csv',
    '../JS09/voice.csv',
    '/content/voice.csv',
    '/content/drive/MyDrive/voice.csv',
    GITHUB_URL,
]
DATA_PATH = next(
    (p for p in CANDIDATES if p.startswith('http') or os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError('voice.csv tidak ditemukan.')
df = pd.read_csv(DATA_PATH)
print('Dataset:', DATA_PATH)

# 2. Preprocessing Data
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])  # male/female menjadi numerik

X = df.drop('label', axis=1)
y = df['label']

# 3. Konfigurasi Eksperimen
splits = {'70:30 (Train: 70, Test: 30)': 0.3, '80:20 (Train: 80, Test: 20)': 0.2}
kernels = ['linear', 'poly', 'rbf']

results = []

# 4. Iterasi Pelatihan dan Pengujian Model
for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # Scaling dihitung dari data latih saja agar tidak terjadi data leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for kernel in kernels:
        svm_model = SVC(kernel=kernel, random_state=42)
        svm_model.fit(X_train_scaled, y_train)

        y_pred = svm_model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': split_name,
            'Kernel SVM': kernel.capitalize(),
            'Akurasi': round(acc * 100, 2)
        })

# 5. Menampilkan Hasil dalam Bentuk Tabel
df_results = pd.DataFrame(results)
print('--- TABEL PERFORMANSI KLASIFIKASI SVM ---')
print(df_results.to_string(index=False))

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. FUNGSI EKSTRAKSI FITUR HISTOGRAM & LOAD DATA
# ==========================================
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)
    if image is None:
        return None
    image = cv2.resize(image, (128, 128))
    hist = cv2.calcHist([image], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

def load_data_from_split(split_dir):
    data = []
    labels = []
    classes = ['day', 'night']

    for label_name in classes:
        folder_path = os.path.join(split_dir, label_name)
        if not os.path.exists(folder_path):
            print(f'Peringatan: folder {folder_path} tidak ditemukan.')
            continue

        for filename in sorted(os.listdir(folder_path)):
            img_path = os.path.join(folder_path, filename)
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                hist = extract_color_histogram(img_path)
                if hist is not None:
                    data.append(hist)
                    labels.append(label_name)

    return np.array(data), np.array(labels)

# ==========================================
# 2. LOKASI DATASET (mendukung training/trainning)
# ==========================================
CANDIDATES = [
    'images',
    '/content/images',
    '/content/drive/MyDrive/images',
    '/content/drive/MyDrive/images/images',
    '/content/drive/MyDrive/dataset ml/images',
    '/content/drive/MyDrive/dataset ml /images',
    '/content/drive/MyDrive/dataset ml /images /images',
]
base_path = next((p for p in CANDIDATES if os.path.isdir(p)), None)
if base_path is None:
    raise FileNotFoundError('Folder images tidak ditemukan.')

train_dir = None
for name in ('training', 'trainning'):
    candidate = os.path.join(base_path, name)
    if os.path.isdir(candidate):
        train_dir = candidate
        break

test_dir = os.path.join(base_path, 'test')

if train_dir is None or not os.path.isdir(test_dir):
    raise FileNotFoundError(f'Folder training/test tidak lengkap di {base_path}')

print('Lokasi dataset:', base_path)
print('Folder training:', train_dir)
print('Folder test    :', test_dir)

# ==========================================
# 3. MUAT SEMUA DATA, LALU SPLIT ULANG 80:20
# ==========================================
print('Memuat data training...')
X_train_raw, y_train_raw = load_data_from_split(train_dir)
print('Memuat data test...')
X_test_raw, y_test_raw = load_data_from_split(test_dir)

X_all = np.vstack([X_train_raw, X_test_raw])
y_all = np.concatenate([y_train_raw, y_test_raw])

if len(X_all) == 0:
    raise ValueError('Tidak ada gambar yang berhasil dimuat.')

le = LabelEncoder()
y_all = le.fit_transform(y_all)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print(f'Total gambar : {len(X_all)}')
print(f'Data training: {len(X_train)} (80%)')
print(f'Data test    : {len(X_test)} (20%)')

# ==========================================
# 4. FEATURE SCALING
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 5. SVM KERNEL RBF & HYPERPARAMETER TUNING
# ==========================================
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

svc_rbf = SVC(kernel='rbf', random_state=42)

grid_search = GridSearchCV(
    estimator=svc_rbf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print('\nSedang melakukan proses training dan hyperparameter tuning...')
grid_search.fit(X_train_scaled, y_train)

print('\n--- HASIL TUNING HYPERPARAMETER ---')
print('Parameter Terbaik :', grid_search.best_params_)
print(f'Akurasi Terbaik (CV): {grid_search.best_score_ * 100:.2f}%')

# ==========================================
# 6. EVALUASI AKHIR PADA DATA TEST (RASIO 80:20)
# ==========================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred)

print('\n--- EVALUASI AKHIR PADA DATA TEST ---')
print(f'Akurasi Pengujian Akhir: {final_accuracy * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))